In [ ]:
# Check what device I am running on, and I want to make sure I am running on TPU 
import torch

def check_gpu():
    """Check GPU availability"""
    if torch.cuda.is_available():
        print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
        return torch.device("cuda")
    else:
        print("📱 Using CPU")
        return torch.device("cpu")

device = check_gpu()

In [ ]:
# Install bitsandbytes for quantisation 
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install --upgrade transformers

Use these models: https://huggingface.co/google/gemma-7b
<br/>
Second model to test: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct

In [ ]:
# Diagnostic check
import sys
print(f"Python version: {sys.version}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")

# Check if bitsandbytes can be imported
try:
    import bitsandbytes as bnb
    print(f"✅ bitsandbytes imported successfully: {bnb.__version__}")
except ImportError as e:
    print(f"❌ Failed to import bitsandbytes: {e}")

# Check if transformers can detect bitsandbytes
try:
    from transformers.utils.quantization_config import is_bitsandbytes_available
    print(f"is_bitsandbytes_available(): {is_bitsandbytes_available()}")
except Exception as e:
    print(f"Error checking bitsandbytes availability: {e}")

# Check if bitsandbytes has CUDA support
try:
    from bitsandbytes.cextension import COMPILED_WITH_CUDA
    print(f"bitsandbytes compiled with CUDA: {COMPILED_WITH_CUDA}")
except Exception as e:
    print(f"Error checking CUDA compilation: {e}")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
from dotenv import load_dotenv
import os
import torch
import bitsandbytes as bnb

# Load environment variables
load_dotenv()

# Quantisation 
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

# Get token from environment variable
hf_token = os.getenv('HF_TOKEN')

# Trying to run inference on Llama 3.1 model 
# Add device mapping and quantization
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    model_id, 
    token=hf_token, 
    padding_side='left', 
    

)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=hf_token,
    device_map="auto",           # Auto device placement
    quantization_config=bnb_config
)

In [ ]:
import torch

def get_model_size(model):
    param_size = 0
    buffer_size = 0
    
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    
    total_size = (param_size + buffer_size) / 1024**3  # GB
    
    print(f"Parameters: {param_size / 1024**3:.2f} GB")
    print(f"Buffers: {buffer_size / 1024**3:.2f} GB") 
    print(f"Total: {total_size:.2f} GB")
    
    return total_size

# Check your model size
model_size_gb = get_model_size(model)

In [ ]:
# Add these functions after your model loading code
import torch


# Generating text 

# the inputs 
prompt = "The weather today is"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Define the hook to be attached 
def hook_extract_vector(module, input, output):
    # input is a tuple, output is a tuple
    # we want the hidden states from the last layer
    
    return output[0]

# Does not calcualte the gradients
with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            # Temperature sampling for creativity
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

        # Retrieve the output for a specific layer here
    
    # Decode only the new tokens (not the input)
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


def generate_text(prompt, max_new_tokens=100, temperature=0.7, top_p=0.9):
    """
    Generate text from a prompt
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            # Temperature sampling for creativity
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode only the new tokens (not the input)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

def get_full_response(prompt, max_new_tokens=100, temperature=0.7, top_p=0.9):
    """
    Get full response including the original prompt
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    full_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return full_response

# Test it out
print("=" * 50)
print("TESTING TEXT GENERATION")
print("=" * 50)

# Simple test
prompt = "The weather today is"
response = generate_text(prompt, max_new_tokens=50)
print(f"Prompt: {prompt}")
print(f"Response: {response}")
print()

# Another test
prompt = "I feel sad because"
response = generate_text(prompt, max_new_tokens=50)
print(f"Prompt: {prompt}")
print(f"Response: {response}")